In [54]:
from pufferlib.ocean.breakout.breakout import test_performance, Breakout
help(test_performance)
import numpy as np
import time 

Help on function test_performance in module pufferlib.ocean.breakout.breakout:

test_performance(timeout=10, atn_cache=1024)



In [55]:
test_performance()

SPS: %f 6067402.789719708


In [56]:
import pufferlib.vector

In [57]:
help(pufferlib.vector.make)

Help on function make in module pufferlib.vector:

make(env_creator_or_creators, env_args=None, env_kwargs=None, backend=<class 'pufferlib.pufferlib.PufferEnv'>, num_envs=1, seed=0, **kwargs)



In [ ]:
env = pufferlib.vector.make(Breakout)



In [59]:
import inspect
from pufferlib.ocean.breakout.breakout import Breakout
print(inspect.signature(Breakout.__init__))

(self, num_envs=1, render_mode=None, frameskip=4, width=576, height=330, paddle_width=62, paddle_height=8, ball_width=32, ball_height=32, brick_width=32, brick_height=12, brick_rows=6, brick_cols=18, continuous=False, log_interval=128, buf=None, seed=0)


In [60]:
def make_action_buffer(n_actions, n_envs, action_buffer_size):
    return np.random.randint(0, n_actions, size=(action_buffer_size, n_envs))

In [107]:
def do_bench(n_envs,n_iters,action_buffer_size):
  env = pufferlib.vector.make(
    Breakout,
    env_kwargs={"num_envs": n_envs},
    num_envs = 16,# C-level vectorization
    backend=pufferlib.vector.Multiprocessing
  )
  obs, _ = env.reset()
  env_action_buffer = make_action_buffer(3,n_envs*16,action_buffer_size)
  start = time.perf_counter()
  for i in range(n_iters): 
    actions = env_action_buffer[i%action_buffer_size] 
    obs,rewards,_,_,_ = env.step(actions)
  end = time.perf_counter()
  
  steps = n_iters*n_envs 
  return steps/(end-start)
  

In [134]:
n_envs = 1024
n_iters = 100_000 
action_buffer_size = 2048

steps_per_second = do_bench(n_envs, n_iters, action_buffer_size)

In [135]:
print(steps_per_second/1e6)


2.9828630457830574


In [92]:
import os
print(f"CPU cores available: {os.cpu_count()}")

# Check if pufferlib respects this
import pufferlib
print(dir(pufferlib.vector))

CPU cores available: 32
['CLOSE', 'GymnasiumPufferEnv', 'INFO', 'MAIN', 'Multiprocessing', 'PettingZooPufferEnv', 'PufferEnv', 'RECV', 'RESET', 'Ray', 'SEND', 'STEP', 'Serial', 'T', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_worker_process', 'autotune', 'check_envs', 'gymnasium', 'make', 'make_seeds', 'np', 'psutil', 'pufferlib', 'recv_precheck', 'reset', 'send_precheck', 'set_buffers', 'step', 'time']
